# Administrative boundaries (OCHA COD-AB)

This notebook downloads OCHA Common Operational Dataset administrative boundaries (COD-AB) for
Senegal as a Shapefile via the `hdx` backend, then reads them with geopandas. The query is a
**live** public download (no credentials), wrapped in `try`/`except` so it degrades gracefully
under offline `nbval-lax`.

## Setup

Imports and the output directory. `earthlens` provides the unified `EarthLens` entry point;
geopandas is imported later, only when we actually read the downloaded Shapefile.

In [ ]:
from pathlib import Path

from earthlens.core import EarthLens

OUT_DIR = Path('hdx_output')
OUT_DIR.mkdir(exist_ok=True)

## Download the boundaries

We build the request first — source, dataset id, the `SHP` format, and the output directory.
Keeping construction on its own line makes each step easy to read and re-run.

In [ ]:
boundaries = EarthLens(
    data_source='hdx',
    dataset='cod-ab-senegal',
    variables=['SHP'],
    path=str(OUT_DIR),
)

`download()` fetches the Shapefile to `OUT_DIR` and returns the list of written paths. The
`try`/`except` lets the notebook skip cleanly when the live HDX query is unavailable (e.g.
offline).

In [ ]:
paths = None
try:
    paths = boundaries.download(progress_bar=False)
    print('downloaded:', [Path(p).name for p in paths])
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## Read the Shapefile

If the download succeeded, read the first written path into a `GeoDataFrame` and preview a few
rows along with the feature count and CRS.

In [ ]:
try:
    import geopandas as gpd

    if paths:
        gdf = gpd.read_file(paths[0])
        print('features:', len(gdf), '| crs:', gdf.crs)
        display(gdf.head())
except Exception as exc:
    print(f'skipped read: {type(exc).__name__}: {exc}')